In [2]:
#need to get neuron indices for each l2 activation in each digit
#plan is to group common neurons for loopy numbers
#need to work out which weights are significant
#can't just select all bc everything pays weak attention to everything else
#can say that if this groups exists, it will have either all positive or all negative activations across all loopy numbers 

#plan
#get activations for all l2 neurons for all numbers, loopy and not
#get all same sign activations for only loopy numbers
#once i have isolated activation group for all loopy numbers, look at l1 attention for this group. 
#is it loopy?

#plan
#1) get average activations for each digit + neuron indices for each activation
#2) find common signed activations for only loopy numbers
#3) then work back from those neurons to first layer
#4) then visualise and find a loop

#except i already did this in interp 1

In [16]:
import torch.nn as nn
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = torch.device("cpu") #bc i'm doing this in aws, no gpu needed
transform = transforms.ToTensor() #this is a torchvision package for casting images to torch.Tensor
test_ds = datasets.MNIST(root="", train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=56, shuffle=False)
model = nn.Sequential(
    nn.Flatten(), #unsurprisingly, this flattens data 0
    nn.Linear(28*28, 256), #linear layer 1
    nn.ReLU(), # 2
    nn.Linear(256, 128), #squash down 3
    nn.ReLU(), # 4
    nn.Linear(128, 10) #output 5
).to(device)
#each linear layer is an n x n matrix. the first layer is 28*28 whatever values multiplied down to 256 values
model.load_state_dict(torch.load("mlp_mnist_state_dict.pt", map_location=device))

def get_activations_for_all_digits(model):
    acts = {}
    digit_activations = {x: [] for x in range(10)}

    def save_activation(name):
        def hook(module, inp, out):
            acts[name] = out.detach()
        return hook

    h1 = model[2].register_forward_hook(save_activation("h1"))#create forward hook on this and call it a1
    z2 = model[3].register_forward_hook(save_activation("z2"))
    logits = model[5].register_forward_hook(save_activation("logits"))

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)

            _ = model(x)
            
            h1 = acts.pop("a1")#these are of length(batch) and contain activations for each value in y 
            z2 = acts.pop("a2")
            logits = acts.pop("logits")


            for i in range(len(y)):#iterate through y
                digit = y[i].item()#cast tensor to python int
                digit_activations[digit].append([h1[i], z2[i], logits[i]])#take slices out of a1, a2, and a3

        h1.remove()#remove hooks when done
        z2.remove()
        logits.remove()

        return digit_activations

In [17]:
results = get_activations_for_all_digits(model)

In [18]:
avg_activations = {}#this is a dict

for digit in range(10):#for every value
    #this is just nested iterators
    layer_2_avg = torch.stack([act[1] for act in results[digit]]).mean(dim=0)#takes value from first iterator. then gets all activations for that digit and puts into a 128 x whatever matrix. then averages across whatever
    #result is iterated
    avg_activations[digit] = layer_2_avg

In [20]:
#print(digit_to_activations)
#now i need to make two boolean masks for each. one positive and one negative

In [19]:
positive_masks = {digit: acts > 0 for digit, acts in avg_activations.items()}
negative_masks = {digit: acts < 0 for digit, acts in avg_activations.items()}

In [21]:
loopy_digits = (0, 6, 9, 8)
#all_loopy_positive = positive_masks[0] & positive_masks[6] & positive_masks[8] & positive_masks[9]
#all_loopy_negative = negative_masks[0] & negative_masks[6] & negative_masks[8] & negative_masks[9]
all_loopy_positive = torch.stack([positive_masks[d] for d in loopy_digits]).all(dim=0)
all_loopy_negative = torch.stack([negative_masks[d] for d in loopy_digits]).all(dim=0)

In [30]:
print(all_loopy_positive)#this shows which average activations across all loopy digits are positives per neuron

tensor([ True, False, False, False, False, False, False,  True,  True, False,
        False, False, False, False,  True,  True,  True,  True, False, False,
        False, False, False,  True,  True, False,  True,  True, False,  True,
        False,  True, False, False,  True,  True, False, False, False, False,
         True,  True, False,  True, False, False, False,  True, False, False,
         True, False, False,  True,  True, False,  True, False, False, False,
        False, False, False, False,  True, False,  True,  True, False, False,
        False,  True,  True, False, False, False,  True, False, False, False,
        False,  True,  True,  True, False,  True,  True, False, False, False,
         True,  True, False, False, False, False,  True, False, False, False,
         True, False,  True, False, False, False,  True, False, False, False,
        False, False,  True, False, False, False,  True,  True, False, False,
        False, False,  True, False,  True,  True, False, False])

In [23]:
#here i encounter the slop values problem. nothing is 0. from my cosine similarity experiments, i know that if the loopy feature exists, it is quite insignificant. 
#to summarise, the loopy vector is a group of neurons that activate when a loop is closed
#i'll start by taking the low hanging fruit and investigating false

In [24]:
false_indices = torch.where(~all_loopy_positive)[0]#this is the indices of every neuron which is negative when averaged over loopy numbers

In [25]:
print(false_indices)
#these results are weird, why are they so similar?

tensor([  1,   2,   3,   4,   5,   6,   9,  10,  11,  12,  13,  18,  19,  20,
         21,  22,  25,  28,  30,  32,  33,  36,  37,  38,  39,  42,  44,  45,
         46,  48,  49,  51,  52,  55,  57,  58,  59,  60,  61,  62,  63,  65,
         68,  69,  70,  73,  74,  75,  77,  78,  79,  80,  84,  87,  88,  89,
         92,  93,  94,  95,  97,  98,  99, 101, 103, 104, 105, 107, 108, 109,
        110, 111, 113, 114, 115, 118, 119, 120, 121, 123, 126, 127])
torch.Size([128, 256])


In [29]:
print(model[3].weight.shape)

torch.Size([128, 256])


In [26]:
#i need to 
false_weights = model[3].weight.detach().cpu()[false_indices,:]
print(false_weights.shape)

torch.Size([82, 256])


In [28]:
#i need to now inspect the
print(len(model[1].weight.detach().cpu()[1]))

784
